Language: **English (default)** | [Polski](lab.ipynb)

# Lab 08 -- a content-based image retrieval system (EuroSAT)

Companion material to [`README.md`](README.md). A different dataset than
the lecture (EuroSAT, not Caltech101) and a different task: not
classification, only **retrieval** -- a query (image) -> a ranking of the
most similar images from a database (gallery).

In [ ]:
import contextlib
import io

import matplotlib.pyplot as plt  # noqa: F401 - used in section 8 (visualization)
import torch
import torch.nn as nn
import torchvision.datasets as tvd
from torchvision.models import ResNet18_Weights, resnet18

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Using device: {device}")

## 1. Data: gallery and queries

In [ ]:
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    full_ds = tvd.EuroSAT(root="data_cache", download=True)

CLASSES = full_ds.classes
GALLERY_PER_CLASS = 40
QUERY_PER_CLASS = 10

class_to_indices = {c: [] for c in range(10)}
for idx, (_, label) in enumerate(full_ds):
    if len(class_to_indices[label]) < GALLERY_PER_CLASS + QUERY_PER_CLASS:
        class_to_indices[label].append(idx)
    if all(len(v) >= GALLERY_PER_CLASS + QUERY_PER_CLASS for v in class_to_indices.values()):
        break

gallery_indices, gallery_labels = [], []
query_indices, query_labels = [], []
for c in range(10):
    idxs = class_to_indices[c]
    gallery_indices += idxs[:GALLERY_PER_CLASS]
    gallery_labels += [c] * GALLERY_PER_CLASS
    query_indices += idxs[GALLERY_PER_CLASS:GALLERY_PER_CLASS + QUERY_PER_CLASS]
    query_labels += [c] * QUERY_PER_CLASS

print(f"gallery: {len(gallery_indices)} images, queries: {len(query_indices)} images")

In [ ]:
# TODO: verify that gallery_indices and query_indices have NO elements in
# common (hint: set() and the & operator)
raise NotImplementedError("TODO: verify gallery/query disjointness")

In [ ]:
weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

gallery_imgs = torch.stack([preprocess(full_ds[i][0]) for i in gallery_indices])
query_imgs = torch.stack([preprocess(full_ds[i][0]) for i in query_indices])
gallery_y = torch.tensor(gallery_labels)
query_y = torch.tensor(query_labels)
print(f"gallery_imgs={tuple(gallery_imgs.shape)}  query_imgs={tuple(query_imgs.shape)}")

## 2. Extracting embeddings

In [ ]:
import time

model = resnet18(weights=weights)
# TODO: set the model to a mode that does NOT update BatchNorm statistics
# (see module 7: requires_grad is NOT the same as model.eval())
raise NotImplementedError("TODO: model mode")

feat_extractor = nn.Sequential(*list(model.children())[:-1])

start = time.time()
# TODO: compute gallery_feats and query_feats (flattened feat_extractor output)
# UNDER a context that does NOT build an autograd graph (torch.no_grad() or
# torch.inference_mode() -- see module 7 for the difference)
raise NotImplementedError("TODO: extracting embeddings")
extract_time = time.time() - start
print(f"gallery_feats={tuple(gallery_feats.shape)}  time: {extract_time:.2f}s")  # noqa: F821

**Task:** if the database were $1000\times$ larger, you would save the
embeddings to disk. Under which **four** conditions would such a cache
stop being correct?

_(Your answer here.)_

## 3. L2 normalization

In [ ]:
def l2_normalize(X):
    # TODO: return X divided by the norm of EACH row (with a safeguard
    # against dividing by zero -- see the lecture)
    raise NotImplementedError("TODO: l2_normalize")


gallery_n = l2_normalize(gallery_feats)  # noqa: F821 - defined in section 2
query_n = l2_normalize(query_feats)  # noqa: F821 - defined in section 2
norms = gallery_n.norm(dim=1)
print(f"norms after normalization: min={norms.min():.4f} max={norms.max():.4f}")
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-4)
print("OK: all embeddings have norm ~1 after normalization.")

## 4. Top-k search

Write the nearest-neighbor ranking by cosine similarity yourself -- no
ready-made library function.

In [ ]:
def search(query_embedding, gallery_embeddings, k):
    """Returns the indices of the k most similar vectors from the gallery
    (cosine similarity), sorted from the best. query_embedding: (d,).
    gallery_embeddings: (N, d)."""
    # TODO:
    # 1. L2-normalize query_embedding and gallery_embeddings
    # 2. compute the cosine similarity between the query and EVERY vector
    #    in the gallery (after L2 normalization this is a plain dot product)
    # 3. return the indices of the k largest values, sorted descending
    raise NotImplementedError("TODO: search")

**Verification on a hand-constructed example with a precisely known result:**

In [ ]:
toy_gallery = torch.tensor([
    [1.0, 0.0],   # idx 0, class A
    [0.9, 0.1],   # idx 1, class A
    [0.8, 0.2],   # idx 2, class A
    [0.0, 1.0],   # idx 3, class B
    [0.1, 0.9],   # idx 4, class B
    [0.5, 0.5],   # idx 5, class A (ambiguous direction)
])
toy_gallery_y = torch.tensor([0, 0, 0, 1, 1, 0])
toy_query = torch.tensor([1.0, 0.0])  # exactly idx 0's direction

toy_result = search(toy_query, toy_gallery, k=3)
print("search() result:", toy_result.tolist())
expected = torch.tensor([0, 1, 2])  # computed by hand: the largest cosine similarities
msg = f"expected {expected.tolist()}, got {toy_result.tolist()}"
assert torch.equal(toy_result, expected), msg
print("OK: search() returns exactly the expected ranking on the control example.")

## 5. Precision@k

In [ ]:
def precision_at_k(query_embeddings, query_labels, gallery_embeddings, gallery_labels, k):
    """Average Precision@k over all queries. Relevance = class agreement
    (a simplification -- see README)."""
    # TODO:
    # 1. for each query: find the top-k in the gallery (use search())
    # 2. compute what fraction of those k have a label matching the query's label
    # 3. average over ALL queries (explicit denominator: number of queries)
    raise NotImplementedError("TODO: precision_at_k")

In [ ]:
toy_query_label = torch.tensor([0])
toy_precision = precision_at_k(
    toy_query.unsqueeze(0), toy_query_label, toy_gallery, toy_gallery_y, k=3
)
print("Precision@3 on the control example:", toy_precision)
assert toy_precision == 1.0, "expected 1.0 (all top-3 are class A)"
print("OK: precision_at_k() matches the manual computation.")

In [ ]:
for k in [1, 5, 10]:
    p = precision_at_k(query_n, query_y, gallery_n, gallery_y, k)
    print(f"Precision@{k} (ResNet18, cosine, normalized): {p:.3f}")

## 6. Baseline: pixel distance

In [ ]:
# TODO: compute Precision@1/5/10 for RAW, flattened pixels instead of
# ResNet18 embeddings (the same precision_at_k function, the same
# gallery/query split -- only a different input representation, L2-normalized)
raise NotImplementedError("TODO: pixel baseline")

## 7. Effect of normalization and metric

In [ ]:
# TODO:
# 1. variant 1: cosine similarity on L2-normalized embeddings (you already
#    have this from section 5 - p_cosine_norm)
# 2. variant 2: write dot_search() -- a ranking by plain dot product on
#    UNnormalized gallery_feats/query_feats (no l2_normalize)
# 3. variant 3: write euclidean_search() -- a ranking by Euclidean distance
#    on UNnormalized gallery_feats/query_feats
# 4. compute Precision@5 for variants 2 and 3 (you can write analogous
#    precision_at_k_dot/precision_at_k_euclidean functions, or generalize search())
# 5. print all three results side by side
raise NotImplementedError("TODO: effect of metric and normalization")

_(Your answer here.)_

## 8. Visualization and error analysis

In [ ]:
# TODO:
# 1. write a show_query_and_results(query_idx_in_query_set, k=5) function
#    that draws the query and its top-k gallery results (image + label +
#    whether it was a hit) -- remember to undo the preprocessing
#    normalization for display
# 2. compute Precision@5 SEPARATELY for each query (not averaged), find
#    the query with the highest and lowest score
# 3. call show_query_and_results() for both
# 4. (systematically) count which (true class, retrieved class) pairs
#    appear most often among the top-5 errors across ALL queries
raise NotImplementedError("TODO: visualization and error analysis")

_(Your answer here.)_

## 9. Conclusions

Answer, justified with actual numbers from sections 5-8.

**1. By how much do ResNet18 embeddings beat the raw-pixel baseline? Is this surprising, given a domain distant from ImageNet?**

_(Your answer here.)_

**2. Did the choice of metric/normalization have a large effect? Would you normalize embeddings when building such a system from scratch?**

_(Your answer here.)_

**3. Label agreement is a simplified relevance measure. Did you find an example where this simplification fails?**

_(Your answer here.)_

**4. What is at least one limitation of this experiment that prevents you from generalizing the conclusions to any arbitrary image-retrieval system?**

_(Your answer here.)_